In [41]:
!pip3 install -r requirements.txt

  Cloning https://github.com/evfro/polara.git to /private/var/folders/0y/g49_sx310b90y_dbbtzchfpc0000gn/T/pip-install-j89uh_ol/polara_0d5e059ff4ef492d94943d32f372e12a
  Running command git clone --filter=blob:none --quiet https://github.com/evfro/polara.git /private/var/folders/0y/g49_sx310b90y_dbbtzchfpc0000gn/T/pip-install-j89uh_ol/polara_0d5e059ff4ef492d94943d32f372e12a
  Resolved https://github.com/evfro/polara.git to commit 86a5d247335242f31baf8fb68e472b651ae6770f
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [42]:
from typing import Callable, Dict, List, Tuple, Any, Optional

from polara import get_movielens_data


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


import torch
from torch.utils.data import Dataset, DataLoader


from src.data.dataprep import (transform_indices, verify_time_split, reindex_data, temporal_train_test_split)


In [43]:
data = get_movielens_data(include_time=True, get_genres=False, local_file='ml-20m.zip')
items_metadata_ = pd.read_csv('https://raw.githubusercontent.com/evfro/recsys19_hybridsvd/master/data/meta_info_ml1m.csv', sep=';')

In [44]:
train_, test_ = temporal_train_test_split(data, test_last_seconds=data['timestamp'].max() - data['timestamp'].quantile(0.9))

In [45]:
train, data_index = transform_indices(train_, users='userid', items='movieid')
test = reindex_data(test_, data_index, entities=['users', 'items'], filter_invalid=True)

items_metadata = reindex_data(items_metadata_, data_index, entities=['items'], filter_invalid=True)


verify_time_split(train, test)

In [46]:
data_description = dict(
    users = data_index['users'].name,
    items = data_index['items'].name,
    timestamp = 'timestamp',
    feedback = 'rating',
    n_users = len(data_index['users']),
    n_items = len(data_index['items'])
)

In [47]:
class TransfromerTrainDataset(Dataset):
    def __init__(
        self,
        histories: Dict[Any, List[int]],
        max_seq_len: int = 100,
    ) -> None:
        super().__init__()

        self.samples = []

        for uid, history in histories.items():
            indicies = np.arange(0, len(history), max_seq_len)
            splits = np.split(history[:-1], indicies)[1:]
            targets = np.split(history, indicies+1)[1:]
            
            self.samples.extend(
                {
                    'uid': uid,
                    'history': list(s),
                    'targets': list(t),
                    'length': len(s)
                }
                for s, t in zip(splits, targets) if len(s) > 0 and len(t) > 0
            )

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        return self.samples[idx]

In [48]:
class TransfromerEvalDataset(Dataset):
    def __init__(
        self,
        histories: Dict[Any, List[int]],
        targets: Dict[Any, List[int]],
        max_seq_len: int = 100,
    ) -> None:
        super().__init__()

        
        targets_uids = targets.keys()
        self.samples = [
            {
                'uid': uid,
                'history': history[-max_seq_len:],
                'length': len(history)
            }
            for uid, history in histories.items() if uid in targets_uids and history
        ]

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        return self.samples[idx]

In [ ]:
TRAIN_BATCH_SIZE = 128
EVAL_BATCH_SIZE = 128

In [ ]:
def collate_fn(batch: List[Dict[str, Any]]) -> Dict[str, Any]:

    uids = torch.stack([torch.tensor(b['uid'], dtype=torch.long) for b in batch])
    lengths = torch.stack([torch.tensor(b['length'], dtype=torch.long) for b in batch])
    histories = torch.concat([torch.tensor(b['history'], dtype=torch.long) for b in batch])

    res = {
        'uid': uids,
        'length': lengths,
        'history': histories
        }

    if batch[0].get('targets', None):
        targets = torch.concat([torch.tensor(b['targets'], dtype=torch.long) for b in batch])
        res['targets'] = targets

    return res

In [51]:
histories = train.sort_values([data_description['users'], data_description['timestamp']], ascending=False).groupby(by=data_description['users'])[data_description['items']].apply(list).to_dict()
targets = test.sort_values([data_description['users'], data_description['timestamp']], ascending=False).groupby(by=data_description['users'])[data_description['items']].apply(list).to_dict()

In [52]:
ml_train_dataset = TransfromerTrainDataset(histories=histories)
ml_eval_dataset = TransfromerEvalDataset(histories=histories, targets=targets)

In [53]:
ml_train_dataloader = DataLoader(
    dataset=ml_train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    drop_last=True
)

yambda_eval_dataloader = DataLoader(
    dataset=ml_eval_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    drop_last=False
)